In [2]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import PBSCluster
import matplotlib.pyplot as plt
import numpy as np

In [3]:
cluster = PBSCluster(
    cores=1,
    memory='40GB',
    processes=1,
    queue='casper',
    local_directory='$TMPDIR',
    account='P93300313',
    walltime='4:00:00'
)
cluster.scale(jobs=10)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.185:39475,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


In [4]:
path = '/glade/work/acruz/Caribbean_Heat_data/ERA5/'
hi_ds = xr.open_dataset(path+'hourly_HI.nc')
u_ds  = xr.open_zarr(path+'U10')
v_ds  = xr.open_zarr(path+'V10')

/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/pyproj/network.py:59: UserWarning: pyproj unable to set PROJ database path.
  _set_context_ca_bundle_path(ca_bundle_path)
/glade/u/home/acruz/.conda/envs/Caribe_Heat_AN_2026/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
u_ds = u_ds['VAR_10U'].sel(time=slice('1940', '2025'))
u_ds

<xarray.DataArray 'VAR_10U' (time: 753888, latitude: 82, longitude: 121)> Size: 30GB
dask.array<open_dataset-VAR_10U, shape=(753888, 82, 121), dtype=float32, chunksize=(62824, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [6]:
v_ds = v_ds['VAR_10V'].sel(time=slice('1940', '2025'))
v_ds

<xarray.DataArray 'VAR_10V' (time: 753888, latitude: 82, longitude: 121)> Size: 30GB
dask.array<open_dataset-VAR_10V, shape=(753888, 82, 121), dtype=float32, chunksize=(62824, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0

In [7]:
# chunk for dask
hi_ds = hi_ds.chunk(time=17166)
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 755304, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2026-02-28T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB dask.array<chunksize=(17166, 82, 121), meta=np.ndarray>

In [8]:
# time range
hi_ds = hi_ds.sel(time=v_ds['time'])
hi_ds

<xarray.Dataset> Size: 30GB
Dimensions:    (time: 753888, latitude: 82, longitude: 121)
Coordinates:
  * time       (time) datetime64[ns] 6MB 1940-01-01 ... 2025-12-31T23:00:00
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Data variables:
    HI_hourly  (time, latitude, longitude) float32 30GB dask.array<chunksize=(17166, 82, 121), meta=np.ndarray>

In [9]:
# make hour dim lazily
hi_ds_coarse = hi_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
u_ds_coarse = u_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))
v_ds_coarse = v_ds.coarsen(time=24, boundary='trim').construct(time=('day', 'hour'))

# get hour where max happens
hidmax_idx = hi_ds_coarse['HI_hourly'].argmax(dim='hour')

def sel_at_idx(ds, idx_ds):
    # add hour dimension to indexing array
    idx_ds = np.expand_dims(idx_ds, axis=-1)
    # select along hour axis and remove hour axis
    return np.take_along_axis(ds, idx_ds, axis=-1).squeeze(axis=-1)

In [10]:
u_peakHI = xr.apply_ufunc(sel_at_idx, u_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[u_ds_coarse.dtype]
                             )
u_peakHI = u_peakHI.rename('U_during_HIdmax')
u_peakHI

<xarray.DataArray 'U_during_HIdmax' (day: 31412, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31412, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [11]:
v_peakHI = xr.apply_ufunc(sel_at_idx, v_ds_coarse, hidmax_idx,
                              input_core_dims=[['hour'], []],
                              output_core_dims=[[]],
                              dask='parallelized',
                              output_dtypes=[v_ds_coarse.dtype]
                             )
v_peakHI = v_peakHI.rename('V_during_HIdmax')
v_peakHI

<xarray.DataArray 'V_during_HIdmax' (day: 31412, latitude: 82, longitude: 121)> Size: 1GB
dask.array<transpose, shape=(31412, 82, 121), dtype=float32, chunksize=(715, 82, 121), chunktype=numpy.ndarray>
Coordinates:
  * latitude   (latitude) float64 656B 7.75 8.0 8.25 8.5 ... 27.5 27.75 28.0
  * longitude  (longitude) float64 968B -89.0 -88.75 -88.5 ... -59.25 -59.0
Dimensions without coordinates: day

In [12]:
timestamps = hi_ds['time'].isel(time=slice(0, None, 24)).dt.floor("1D")
timestamps

<xarray.DataArray 'floor' (time: 31412)> Size: 251kB
array(['1940-01-01T00:00:00.000000000', '1940-01-02T00:00:00.000000000',
       '1940-01-03T00:00:00.000000000', ...,
       '2025-12-29T00:00:00.000000000', '2025-12-30T00:00:00.000000000',
       '2025-12-31T00:00:00.000000000'],
      shape=(31412,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 251kB 1940-01-01 1940-01-02 ... 2025-12-31

In [13]:
u_peakHI = u_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop_vars('day')
v_peakHI = v_peakHI.rename({'day': 'time'}).assign_coords(day=timestamps).drop_vars('day')

In [15]:
u_peakHI.to_netcdf(path+'U10_during_peak_HI.nc', mode='w')

In [16]:
v_peakHI.to_netcdf(path+'V10_during_peak_HI.nc', mode='w')

In [17]:
client.shutdown()